In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.5 Thomas–Fermi: The First Density Functional

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.5",
    title="Thomas–Fermi: The First Density Functional",
    blurb="Movement II opens with 1927's audacious idea: throw away the wavefunction "
    "and work with the density alone. We build the functional-derivative toolkit "
    "the whole movement runs on, solve the universal Thomas–Fermi atom by shooting "
    "(every neutral atom is one curve in disguise, with energy scaling as "
    "Z to the seven-thirds), measure the theory's honest failures on the exact "
    "laboratory, demonstrate Teller's devastating no-binding theorem, and "
    "construct Harriman's orbitals to prove any density is representable. The "
    "idea is right; the kinetic functional is crude; the fix is two notebooks away.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

Movement I ended with wavefunction methods priced: exact is impossible
([§8.1](many-electron-problem.ipynb)), Hartree–Fock misses correlation by
construction ([§8.3](hartree-fock-atoms.ipynb),
[§8.4](hartree-fock-gas.ipynb)). Movement II pursues a different currency
altogether. In 1926–27, before Hartree had orbitals, Thomas and Fermi
{cite}`thomas1927` proposed describing an atom by its *density alone*: three
variables instead of $3N$, with the kinetic energy estimated locally from the
Fermi-gas formula of [§7.9](../07-quantum-statistical-mechanics/fermi-gas-zero-temperature.ipynb).
The idea seems reckless and is in fact prophetic — it is the direct ancestor of
density-functional theory, and the Hohenberg–Kohn theorem of
[§8.6](hohenberg-kohn.ipynb) will prove the reckless part *exactly right*: the
density really does suffice. What fails is only the crude kinetic estimate, and
measuring *how* it fails is this notebook's job.

The itinerary: build the functional-derivative toolkit (the movement's calculus,
certified numerically the way [§0.3](../00-foundations/quadrature-differentiation.ipynb)
certified quadrature); derive the TF functional and reduce every neutral atom to
one universal ODE, solved by shooting with a series-expansion launch (the
singular origin is a genuine numerics lesson) to Baker's slope $-1.588071$;
assemble the marvellous scaling law $E = -0.7687\,Z^{7/3}$ Ha; then the honest
failures, each computed: no shells, an atom without an edge (the chemical
potential of the neutral atom vanishing as a measured power law), a 34% energy
error on the exact laboratory of [§8.2](exact-laboratory.ipynb), and Teller's
theorem — Thomas–Fermi molecules never bind — demonstrated as a computed binding
curve with no minimum. The closing exercise is constructive optimism: Harriman's
explicit orbitals show *any* reasonable density is $N$-representable, clearing
the ground the Hohenberg–Kohn theorems will build on.

> **Conventions (this notebook).** Hartree atomic units. The 3-D Thomas–Fermi
> atom uses the standard scaled variables $r = b\,x$ with
> $b = \tfrac12(3\pi/4)^{2/3} Z^{-1/3} = 0.8853\,Z^{-1/3}$. One-dimensional
> laboratory computations use the [§8.2](exact-laboratory.ipynb) system
> (soft-Coulomb, charge-2 center) with the 1-D Fermi-gas kinetic density
> $\pi^2 n^3/24$. Functional derivatives are checked against `numpy` central
> differences with Gaussian test perturbations; the universal ODE is solved by
> `scipy.integrate.solve_ivp` (`rtol=1e-12`) inside a `scipy.optimize.brentq`
> shooting loop.
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: a closed-form derivative, Baker's slope, the
> $Z^{7/3}$ law, the exact laboratory's energy, a constructive identity. A ✓ is
> strong evidence; a ✗ is a prompt to locate the discrepancy, not an automatic
> verdict.
>
> **Scope.** Thomas–Fermi in its original form: no exchange (Dirac's
> $-c\,n^{4/3}$ addition is named), no gradient corrections (von Weizsäcker
> named), both belonging to the systematic story in Parr & Yang
> {cite}`parryang1989`, Ch. 6, the subject's standard reference. Teller's
> theorem is {cite}`teller1962`; the Harriman construction is
> {cite}`harriman1981`; Lieb's rigorous treatment of TF theory is surveyed in
> Parr & Yang, Ch. 6.

## Theory in brief

### The functional-derivative toolkit

Density-functional theory's calculus is variation with respect to a *function*:
for a functional $F[n]$, the functional derivative $\delta F/\delta n(x)$ is
defined by the first-order response
$F[n + \epsilon\,\eta] - F[n] = \epsilon\!\int (\delta F/\delta n)(x)\,\eta(x)\,dx + O(\epsilon^2)$
for arbitrary smooth $\eta$. Two workhorse cases cover
everything this movement needs (Parr & Yang {cite}`parryang1989`, App. A,
develop the calculus in full):

```{math}
:label: eq-tf-funcderiv
F[n] = \!\int\! f\big(n(x)\big)\,dx \;\Rightarrow\;
\frac{\delta F}{\delta n(x)} = f'\big(n(x)\big),
\qquad
E_H[n] = \tfrac12\!\iint\! n(x)\,w(x,x')\,n(x')
\;\Rightarrow\;
\frac{\delta E_H}{\delta n(x)} = \!\int\! w(x,x')\,n(x')\,dx' .
```

Local integrands differentiate pointwise; the Hartree energy's derivative is the
classical potential of the cloud. Both claims are *checked numerically* below —
the toolkit is certified before it is trusted, exactly as the volume's radial
machinery was in [§8.1](many-electron-problem.ipynb).

### The Thomas–Fermi functional and the universal atom

The gamble of Thomas and Fermi {cite}`thomas1927`: treat each volume element of
an atom as a scrap of uniform electron gas. The
[§7.9](../07-quantum-statistical-mechanics/fermi-gas-zero-temperature.ipynb)
kinetic energy per volume of a gas at density $n$ is
$\tfrac35 n\,\varepsilon_F = C_{\mathrm{TF}}\, n^{5/3}$ with
$C_{\mathrm{TF}} = \tfrac{3}{10}(3\pi^2)^{2/3}$, so

```{math}
:label: eq-tf-functional
E_{\mathrm{TF}}[n] = C_{\mathrm{TF}}\!\int\! n^{5/3}\,d^3r
\;-\; Z\!\int\! \frac{n(\mathbf r)}{r}\,d^3r
\;+\; \tfrac12\!\iint\!\frac{n(\mathbf r)\,n(\mathbf r')}{|\mathbf r-\mathbf r'|} .
```

Minimizing over $n \ge 0$ at fixed $\int n = N$ (a Lagrange multiplier $\mu$,
the chemical potential) gives $\tfrac53 C_{\mathrm{TF}} n^{2/3} = \mu -
v_{\mathrm{eff}}(\mathbf r)$, and for the neutral atom the standard substitution
$v_{\mathrm{eff}} = -Z\Phi(x)/r$, $r = bx$, collapses *every element of the
periodic table* onto one parameter-free boundary-value problem
(Parr & Yang {cite}`parryang1989`, §6.2, carry out the reduction):

```{math}
:label: eq-tf-universal
\Phi''(x) = \frac{\Phi^{3/2}}{\sqrt{x}},
\qquad \Phi(0) = 1,\quad \Phi(\infty) = 0,
\qquad
E_{\mathrm{TF}}(Z) = \frac{3}{7}\,\frac{\Phi'(0)}{b/Z^{-1/3}}\;Z^{7/3}
= -0.7687\,Z^{7/3}\ \text{Ha}.
```

The initial slope $\Phi'(0) = -1.588071$ (Baker's constant) is the one number
the computer must find, and the $Z^{7/3}$ scaling — kinetic, nuclear, and
Hartree terms all conspiring to one power — is the theory's enduring truth:
it is the correct leading term of the exact large-$Z$ energy of atoms.

### What must fail, and what survives

Three built-in pathologies, each computed below. *No shells*: the TF density is
a smooth monotone profile (compare beryllium's two humps in
[§8.3](hartree-fock-atoms.ipynb)) because a local gas knows no quantization.
*No edge and no chemistry-grade energetics*: the neutral TF atom extends to
infinity with $\mu = 0$ (measured below as a power law in the box size), and it
binds no negative ions. And most damning, *no molecules*: Teller
{cite}`teller1962` proved the TF energy of a molecule always exceeds the sum of
its atoms — no chemical bond, ever, at any separation. Yet the framework —
energy from density, minimization with a chemical potential, an effective
potential built from the density itself — survives *unchanged* into modern DFT.
The Hohenberg–Kohn theorems ([§8.6](hohenberg-kohn.ipynb)) will bless the
framework; the Kohn–Sham construction ([§8.7](kohn-sham.ipynb)) will replace
the offending kinetic functional with orbitals; and the last ingredient, the
$N$-representability of densities, is settled *constructively* here by
Harriman's explicit orbitals {cite}`harriman1981`: for any density $n \ge 0$
with $\int n = N$, the phase-twisted orbitals

```{math}
:label: eq-tf-harriman
\varphi_k(x) = \sqrt{\frac{n(x)}{N}}\; e^{\,i k f(x)},
\qquad
f(x) = \frac{2\pi}{N}\int_{-\infty}^{x}\! n(x')\,dx',
\qquad k = 0, \dots, N-1,
```

are exactly orthonormal and sum to $n$: any density can host $N$ electrons.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

B_TF = 0.5 * (3.0 * np.pi / 4.0) ** (2.0 / 3.0)  # the TF length b * Z^(1/3) = 0.8853
C_TF_1D = np.pi**2 / 24.0  # 1-D TF kinetic coefficient


def tf_shoot(slope, x_start=1e-3, x_max=50.0):
    """Integrate the universal Thomas-Fermi equation from a series-expansion start.

    Launches Eq. eq-tf-universal at x_start using the singular series
    Phi = 1 + s x + (4/3) x^(3/2) + (2/5) s x^(5/2) + (1/3) x^3 (the x^(3/2)
    term is why naive integration from x ~ 0 stalls at 4 digits: the origin is
    a genuine singular point of the equation). Negative excursions of Phi are
    clamped to zero inside the right-hand side.

    Parameters
    ----------
    slope : float
        Trial initial slope s = Phi'(0).
    x_start : float, optional
        Series launch point (default 1e-3).
    x_max : float, optional
        End of integration (default 50).

    Returns
    -------
    scipy.integrate.OdeSolution-bearing object
        The full solve_ivp result (dense_output enabled).
    """
    s = slope
    phi0 = (
        1.0
        + s * x_start
        + (4.0 / 3.0) * x_start**1.5
        + 0.4 * s * x_start**2.5
        + (1.0 / 3.0) * x_start**3
    )
    dphi0 = s + 2.0 * np.sqrt(x_start) + s * x_start**1.5 + x_start**2

    def rhs(x, y):
        phi = max(y[0], 0.0)
        return [y[1], phi**1.5 / np.sqrt(x)]

    return solve_ivp(
        rhs, [x_start, x_max], [phi0, dphi0], rtol=1e-12, atol=1e-14, dense_output=True
    )


def tf1d_solve(x, v, n_electrons, iterations=1500, mixing=0.05):
    """Solve the 1-D Thomas-Fermi problem for the laboratory's interaction.

    Minimizes pi^2/24 int n^3 + int v n + (1/2) int int n w n at fixed particle
    number by iterating the stationarity condition (pi^2/8) n^2 = mu - v_eff:
    each sweep rebuilds v_eff = v + w*n, finds mu by bisection on the norm, and
    linearly mixes the new density (the damped fixed point of section 0.2).

    Parameters
    ----------
    x : numpy.ndarray
        Uniform grid in Bohr.
    v : numpy.ndarray
        External potential on the grid (Hartree).
    n_electrons : float
        Particle number the density integrates to.
    iterations : int, optional
        Fixed-point sweeps (default 1500; convergence is slow but safe at
        mixing 0.05).
    mixing : float, optional
        Linear mixing fraction.

    Returns
    -------
    tuple
        ``(n, mu, E)``: converged density, chemical potential, and total
        Thomas-Fermi energy in Hartree.
    """
    h = x[1] - x[0]
    w_kernel = 1.0 / np.sqrt((x[:, None] - x[None, :]) ** 2 + 1.0)
    n = np.full(len(x), n_electrons / (x[-1] - x[0]))
    for _ in range(iterations):
        v_eff = v + h * (w_kernel @ n)
        mu = brentq(
            lambda m: np.trapezoid(
                np.sqrt(np.clip(8.0 * (m - v_eff), 0.0, None)) / np.pi, x
            )
            - n_electrons,
            v_eff.min() - 1.0,
            v_eff.min() + 30.0,
        )
        n = (1.0 - mixing) * n + mixing * np.sqrt(
            np.clip(8.0 * (mu - v_eff), 0.0, None)
        ) / np.pi
    energy = (
        C_TF_1D * np.trapezoid(n**3, x)
        + np.trapezoid(v * n, x)
        + 0.5 * h * np.trapezoid(n * (w_kernel @ n), x)
    )
    return n, mu, energy

## Exercise 1 — The toolkit, certified: functional derivatives

Equation {eq}`eq-tf-funcderiv` makes two claims; both get the treatment every
tool in this course gets before use. The test bed is the
[§8.2](exact-laboratory.ipynb) grid with the density
$n_0(x) = e^{-x^2/2}$ (unnormalized on purpose — the derivative rules hold for
any $n$), the local functional $F[n] = \int n^{5/3}\,dx$, the Hartree functional
with the laboratory's soft kernel $w = 1/\sqrt{(x-x')^2+1}$, and the Gaussian
test perturbation $\eta(x) = e^{-(x-1)^2}$.

**Part a)** Compute the claimed derivatives on the grid: $\tfrac53 n_0^{2/3}$
for the local functional, and $h\,(W \mathbin{@} n_0)$ for the Hartree
functional (the `@` product of the kernel matrix with the density, times the
grid spacing).

**Part b)** Test both against the definition: form the response
$\big(F[n_0 + \epsilon\eta] - F[n_0 - \epsilon\eta]\big)/2\epsilon$ by central
differences at $\epsilon = 10^{-6}$ (`numpy.trapezoid` for every integral) and
compare with $\int (\delta F/\delta n)\,\eta\,dx$. Agreement at $10^{-8}$
certifies the calculus the rest of Movement II runs on.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the calculus holds

Both functional derivatives must match their central-difference responses at
$10^{-8}$ relative: the movement's calculus, certified.

In [ ]:
validate.close(
    resp_loc, pred_loc, "the local-functional derivative 5/3 n^(2/3)", rtol=1e-8
)
validate.close(
    resp_har, pred_har, "the Hartree derivative is the cloud's potential", rtol=1e-8
)

## Exercise 2 — The universal atom, shot down to Baker's constant

Equation {eq}`eq-tf-universal` compresses the periodic table into one curve, and
finding it is a classic shooting problem with a twist at each end. At the
origin the equation is *singular*: the solution is not analytic but carries the
series $\Phi = 1 + sx + \tfrac43 x^{3/2} + \tfrac25 s x^{5/2} + \cdots$, so the
Setup helper `tf_shoot` launches from $x_0 = 10^{-3}$ using the series (naive
integration from $x \approx 0$ stalls near four digits — the measured cost of
ignoring a singular point). At infinity the physical solution decays as
$144/x^3$; trial slopes above the critical one diverge, slopes below crash to
zero at finite $x$, and the separatrix *is* the atom.

**Part a)** Bracket and find the critical slope with `scipy.optimize.brentq` on
the boundary residual $\Phi(x_{\max})$ over $s \in [-1.65, -1.55]$
($x_{\max} = 50$, `xtol=1e-12`). Compare with Baker's constant $-1.5880710$.

**Part b)** Plot the separatrix anatomy: the critical curve with the $144/x^3$
asymptote, and two slightly off-critical trials peeling away to either side —
the sensitivity that makes shooting work.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2 — Baker's constant

The critical slope must reproduce Baker's eight-digit value, and the critical
curve must pass through the classic tabulated values $\Phi(1) = 0.4240$,
$\Phi(5) = 0.07881$, $\Phi(10) = 0.02431$ (Kobayashi's tables, the standard
reference solution) to better than $0.2\%$.

In [ ]:
validate.close(
    slope_baker, -1.5880710, "Baker's initial slope of the universal atom", rtol=1e-6
)
for x_tab, phi_tab in ((1.0, 0.4240), (5.0, 0.07881), (10.0, 0.02431)):
    validate.close(
        float(sol_crit.sol(x_tab)[0]),
        phi_tab,
        f"Phi({x_tab:.0f}) vs the classic tabulation",
        rtol=2e-3,
    )
validate.check(
    residual(slope_baker + 1e-3) > 1.0 and tf_shoot(slope_baker - 1e-3).y[0][-1] <= 0.0,
    "the critical curve is a separatrix",
    "off-critical shots diverge or crash",
)

## Exercise 3 — $Z^{7/3}$: one number for the whole periodic table

The energy assembly of Eq. {eq}`eq-tf-universal` turns Baker's slope into the
total energy of any neutral atom, and the scaled density exposes both the
theory's universality and its first pathology.

**Part a)** Assemble the coefficient $\tfrac37\,(-\Phi'(0))/\bar b$ with
$\bar b = 0.88534$ (plain `numpy` arithmetic) and report
$E_{\mathrm{TF}}(Z) = -0.7687\,Z^{7/3}$ Ha; tabulate it for
$Z = 2, 10, 36$ against the known exact ground energies ($-2.90$, $-128.9$,
$-2753$ Ha). TF *overbinds* every atom, and the error shrinks only slowly with
$Z$ — but the shrinkage is itself quantitative physics: the next term of the
large-$Z$ expansion is Scott's $+Z^2/2$ (the inner electrons' correction to
the statistical picture), so the relative error should fall as
$(Z^2/2)/(0.7687\,Z^{7/3}) = 0.650\,Z^{-1/3}$. Compare the measured errors at
$Z = 10$ and $36$ against that prediction (`numpy` arithmetic): Thomas--Fermi
is the exact leading term of quantum chemistry's large-$Z$ expansion, and its
error is the *next* term, visible in this three-row table.

**Part b)** The universality and the no-shell pathology in one figure: plot the
scaled radial density $D(x) \propto \Phi^{3/2}\sqrt{x}$ (one hump, no shell
structure — compare the two-humped beryllium of
[§8.3](hartree-fock-atoms.ipynb)), and verify the neutral-atom norm
$\int_0^\infty \Phi^{3/2}\sqrt{x}\,dx = 1$ by `numpy.trapezoid` on the critical
solution (the truncation at $x_{\max}$ costs the stated percent).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3 — the scaling law and its honest errors

The coefficient must be $0.7687$; the overbinding must shrink monotonically
along $Z = 2, 10, 36$ (Lieb–Simon), and at $Z = 10$ and $36$ it must match the
Scott-term prediction $0.650\,Z^{-1/3}$ within $10\%$ — the theory's error is
the expansion's next term, measured. The neutral norm must come out 1 within
the stated truncation percent.

In [ ]:
validate.close(E_COEFF, 0.7687, "the Thomas-Fermi energy coefficient", rtol=1e-3)
validate.check(
    errors_tf[2] < -0.30,
    "TF overbinds helium by more than 30%",
    f"{100 * errors_tf[2]:.1f}%",
)
validate.check(
    abs(errors_tf[2]) > abs(errors_tf[10]) > abs(errors_tf[36]),
    "the relative error shrinks monotonically with Z (Lieb-Simon)",
    f"{100 * abs(errors_tf[2]):.1f}% > {100 * abs(errors_tf[10]):.1f}% > {100 * abs(errors_tf[36]):.1f}%",
)
for Z_s in (10, 36):
    validate.close(
        abs(errors_tf[Z_s]),
        0.650 * Z_s ** (-1.0 / 3.0),
        f"the error at Z = {Z_s} equals the Scott-term prediction",
        rtol=0.10,
    )
validate.close(norm_tf, 1.0, "the neutral-atom norm from the critical curve", rtol=1e-2)

## Exercise 4 — Thomas–Fermi meets the exact laboratory

How crude is the local kinetic gamble, in digits? The
[§8.2](exact-laboratory.ipynb) laboratory knows its exact answer, and the 1-D
Thomas–Fermi functional (kinetic density $\pi^2 n^3/24$, the 1-D Fermi-gas
result) can be minimized on the same grid by the Setup helper `tf1d_solve` —
the damped fixed point of [§0.2](../00-foundations/root-finding.ipynb), with
the chemical potential found by bisection at every sweep.

**Part a)** Solve the 1-D TF problem for the laboratory's charge-2 atom with
$N = 2$ on $x \in [-20, 20]$ (801 points) and compare $E_{\mathrm{TF}}$ with
the exact $-2.2386$ Ha and Hartree–Fock's $-2.2245$ Ha from
[§8.2](exact-laboratory.ipynb): the TF error is a *third* of the total energy,
two orders beyond Hartree–Fock's.

**Part b)** The atom without an edge: re-solve on boxes $L = 10, 14, 20, 28$
and fit $\ln\mu$ against $\ln L$ (`numpy.polyfit`). The neutral TF atom's
chemical potential must vanish as the measured power $\mu \sim 1/L$: the cloud
has no boundary, only an ever-receding tail — the analytic $\mu = 0$ of the
infinite neutral atom, observed as a finite-size scaling law. Plot the TF
density against the exact one.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4 — the crudeness, in digits

The 1-D TF energy on the $L = 20$ box is $-1.481$ Ha (its own converged value,
gated), a $34\%$ miss against the exact laboratory — while Hartree–Fock missed
by $0.6\%$; and the chemical potential must vanish with the measured
$L^{-1.0}$ law.

In [ ]:
validate.close(E_tf_lab, -1.4812, "the 1-D TF energy of the laboratory atom", rtol=1e-3)
validate.check(
    abs(E_tf_lab - E_EXACT_LAB) / abs(E_EXACT_LAB) > 0.30,
    "TF misses the laboratory by over 30% where HF missed by 0.6%",
    f"{100 * abs(E_tf_lab - E_EXACT_LAB) / abs(E_EXACT_LAB):.1f}%",
)
validate.close(mu_power, -1.0, "the edgeless atom: mu vanishes as 1/L", rtol=5e-2)

## Exercise 5 — Teller's theorem: no molecules, ever

The heaviest blow to Thomas–Fermi theory is not an error percentage but a
theorem. Teller (1962) {cite}`teller1962` proved that in TF theory the energy
of *any* molecule exceeds the summed energies of its isolated atoms at *every*
nuclear separation: no binding, no chemistry, at all. A theorem that sweeping
deserves a computed instance. The test molecule is the laboratory's "diatomic":
two charge-2 soft-Coulomb centers at $\pm R/2$, four electrons, bare nuclear
repulsion $4/R$ — the TF twin of the genuinely binding molecule
[§8.1](many-electron-problem.ipynb) computed with exact quantum mechanics.

**Part a)** Solve the 1-D TF problem (the Setup helper, $N = 4$, the $L = 12$
box with 481 points) at the separations $R = 1.5, 2.5, 4, 6, 9$ Bohr, add the
nuclear repulsion, and tabulate the binding energy
$E_{\mathrm{TF}}(R) + 4/R - 2E_{\mathrm{TF}}^{\mathrm{atom}}$ (the atomic
reference recomputed on the same box and grid spacing, so truncation biases
cancel).

**Part b)** Plot the binding curve. It must be positive everywhere and decrease
monotonically toward zero: the TF molecule always prefers to fall apart —
contrast the genuine minimum of {numref}`fig-many-electron-bo`. The moral, made
precise by later analysis (Parr & Yang {cite}`parryang1989`, §6.4): binding
lives in the *quantum* kinetic energy's response to density deformation, exactly
what the local $n^{5/3}$ ($n^3$ in 1-D) gamble cannot represent.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — Teller, computed

The binding energy must be positive at every separation and strictly
monotonically decreasing: dissociation is always downhill in Thomas–Fermi
theory.

In [ ]:
validate.check(
    bool(np.all(binding_tf > 0.0)),
    "no binding at any separation (Teller)",
    f"minimum binding {binding_tf.min():+.4f} Ha at R = {R_list[-1]}",
)
validate.check(
    bool(np.all(np.diff(binding_tf) < 0.0)),
    "dissociation is monotonically downhill",
    "binding falls at every step of R",
)

## Exercise 6 — Harriman's orbitals: any density can host $N$ electrons

The movement's constructive close. For density-based theory to make sense, the
innocent question "which densities are allowed?" needs an answer, and Harriman
{cite}`harriman1981` gave a disarmingly explicit one: the phase-twisted
orbitals of Eq. {eq}`eq-tf-harriman` are *exactly* orthonormal for any smooth
$n \ge 0$ integrating to $N$ — the equal-density amplitude makes their overlaps
pure phase integrals, and the accumulated phase $f(x)$, which climbs by $2\pi/N$
per unit of enclosed charge, makes those integrals vanish between different
$k$. Any reasonable density is therefore $N$-representable: the variational
domain of the Hohenberg–Kohn functional in [§8.6](hohenberg-kohn.ipynb) is as
large as one could wish. **Write this one yourself** — the implementation is
the lesson.

**Part a)** For the two-hump target density
$n(x) = \big(e^{-(x-1.5)^2} + e^{-(x+1.5)^2}\big)$, normalized to $N = 3$ on
the $[-10, 10]$ grid, build the phase $f(x)$ by cumulative integration
(`numpy.cumsum` of $n\,h$, scaled by $2\pi/N$, *staggered by half a cell*:
subtract $n\,h/2$ so each grid point carries the antiderivative at its own
location rather than its right edge — the midpoint discipline of
[§0.3](../00-foundations/quadrature-differentiation.ipynb), which upgrades the
discrete orthogonality from first to second order in $h$) and the three
complex orbitals of Eq. {eq}`eq-tf-harriman`.

**Part b)** Verify the construction: the Gram matrix
$G_{jk} = h\sum_x \varphi_j^*\varphi_k$ (a `numpy` conjugate contraction) must
equal `numpy.eye(3)` to the grid's quadrature accuracy ($\sim 10^{-5}$ at this
spacing; the construction is exact in the continuum, and the residual is pure
discretization, falling fourfold per grid doubling), and
$\sum_k |\varphi_k|^2$ must reproduce the target density to rounding. Plot the
target, the winding phase, and the (identical) orbital densities.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — the construction is exact

The Gram matrix must be the identity to the grid's second-order quadrature
accuracy and the summed orbital densities the target at rounding:
$N$-representability is not an existence claim but a formula.

In [ ]:
validate.close(
    gram_err,
    0.0,
    "Harriman orbitals orthonormal to quadrature accuracy",
    rtol=0.0,
    atol=5e-5,
)
validate.close(
    rebuild_err, 0.0, "the orbital densities sum to the target", rtol=0.0, atol=1e-12
)

:::{admonition} With your assistant
:class: tip
Dirac's 1930 refinement adds the local exchange of
[§8.4](hartree-fock-gas.ipynb) to the Thomas–Fermi functional:
$E_{x}[n] = -\tfrac34(3/\pi)^{1/3}\!\int n^{4/3}$. Have your assistant extend
the 3-D energy assembly to Thomas–Fermi–Dirac and re-tabulate $Z = 2, 10, 36$,
then run the check that is yours alone: exchange must *lower* every energy
(each TFD value below its TF counterpart), and the correction must scale as
$Z^{5/3}$ — so its *relative* weight falls as $Z^{-2/3}$, fading for heavy
atoms (`numpy` ratios across your table). The check is yours.
:::

## Notebook summary

The first density functional is now built, solved, and honestly convicted. The
functional-derivative toolkit certified itself against central differences at
$10^{-8}$ (local integrands differentiate pointwise; the Hartree derivative is
the cloud's potential). The universal atom emerged from a shooting problem with
a singular-series launch — naive integration stalls at four digits; the series
start reaches Baker's slope $-1.5880710$ to eight — and the energy assembly
gave $E_{\mathrm{TF}} = -0.7687\,Z^{7/3}$ Ha, overbinding helium by $33\%$ and krypton
still by $19.5\%$ — an error that is itself physics: at $Z = 10$ and $36$ it
tracks the Scott correction's $0.650\,Z^{-1/3}$ (within $10\%$, and at the
percent level for krypton), so TF's mistake is the next term of the
large-$Z$ expansion — with a one-hump
universal density innocent of shells. On the exact laboratory the 1-D TF energy
missed by $34\%$ where Hartree–Fock missed by $0.6\%$, and the neutral atom's
chemical potential vanished as the measured $L^{-1.00}$: an atom with no edge.
Teller's theorem materialized as a computed binding curve, positive and
monotone at every separation — no TF molecule ever binds — and Harriman's
phase-twisted orbitals closed the notebook constructively: Gram matrix the
identity to the quadrature floor $10^{-5}$ (with a half-cell stagger of the
accumulated phase buying second order), densities summing to the target at
rounding, any density able to host $N$ electrons.

## Outlook

- Thomas–Fermi guessed that the density suffices; Hohenberg and Kohn *proved*
  it. [§8.6](hohenberg-kohn.ipynb) runs both proofs as computations and then
  inverts the exact laboratory's density into the potential that generates it —
  the theorem as an algorithm.
- The kinetic functional is the sole culprit of this notebook's failures, and
  the Kohn–Sham insight of [§8.7](kohn-sham.ipynb) is surgical: keep the
  density-functional framework, but compute the kinetic energy from *orbitals*
  (bringing shells, edges, and binding back at a stroke) and push the unknown
  remainder into an exchange-correlation functional seeded by
  [§8.4](hartree-fock-gas.ipynb)'s gas.
- Gradient corrections (von Weizsäcker's $|\nabla n|^2/n$ term) and modern
  orbital-free DFT keep the TF program alive for million-atom simulations where
  even Kohn–Sham is too dear; Parr & Yang {cite}`parryang1989`, Ch. 6–7, open
  that road.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()